# 02 · Text-only RAG + chat templates: scenarios and failures

**Steps:**
1. Load a few annual-report PDFs **as text only**, using `scripts/rag/financial_ingest.py` with the `text` stage. This builds Qdrant `docs_text`, with chunks and tables, dense vectors and number-aware sparse vectors.
2. Retrieve with hybrid search plus rerank, filtered by company and year (`retrieve_for_agent`).
3. Answer with the three chat templates and score every answer against the frozen eval set.

**Scenarios:**

| Scenario | Purpose |
|---|---|
| filtered `k=5` | the intended setup |
| no company/year filter | wrong-company evidence |
| filtered `k=2` | too little evidence |
| a filing with a **garbled text layer** (Chevron FY2022, 52 of 116 pages) | text-only RAG cannot read it |

**Runtime:** a GPU runtime is optional; text embeddings use MiniLM on the GPU if present.

In [ ]:
# 0. Setup. On Colab: clones the repo (branch below) and installs packages. Locally: run from inside the repo.
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/rama159065/Capstone.git"
BRANCH = "finance_report_qa_agent"
IN_COLAB = "google.colab" in sys.modules


def colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if IN_COLAB:
    ROOT = Path("/content/Capstone")
    if not ROOT.exists():
        token = colab_secret("GITHUB_TOKEN")          # needed while the repo is private
        url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client>=1.12,<2", "pymupdf>=1.24,<2",
                    "pydantic>=2", "pandas", "sentence-transformers"], check=True)
    for key in ("LLM_API_BASE", "LLM_API_KEY", "LLM_MODEL", "SEC_USER_AGENT"):
        if not os.getenv(key) and colab_secret(key):
            os.environ[key] = colab_secret(key)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("SEC_USER_AGENT", "capstone-research your.name@example.com")   # SEC requires your real e-mail
print("repo:", ROOT, "| Colab:", IN_COLAB)

**LLM.** `NB_LLM=auto` picks, in order:
1. **The provided Track B endpoint**, when `LLM_API_BASE` and `LLM_MODEL` are set (Colab secrets or environment). Grade with this one.
2. **A small Hugging Face stand-in** (`Qwen/Qwen2.5-1.5B-Instruct`), on a Colab GPU.
3. **The deterministic mock** otherwise. It only proves that the pipeline runs.

Force a choice with `os.environ["NB_LLM"] = "hosted" | "transformers" | "mock"` before the config cell.

In [ ]:
from scripts.common import notebook_helpers as nb
from scripts.agents.answer_schema import validate_answer
from scripts.rag import financial_ingest as fi

DOCS = ["AMD_2025", "ABBV_2025", "NVDA_2026", "CVX_2022"]
WORK = Path(os.getenv("NB_WORK", "/content/nb02_text_rag" if IN_COLAB else "tmp/nb02_text_rag"))
EMBEDDING = os.getenv("NB_EMBED", "local")          # local = sentence-transformers MiniLM; hashing = offline, weaker
EMBED_MODEL = os.getenv("NB_EMBED_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
MAX_PAGES = int(os.getenv("NB_MAX_PAGES", "0")) or None
llm, llm_name = nb.make_llm(os.getenv("NB_LLM", "auto"))
rows = nb.pick_filings(DOCS)
pdfs = nb.fetch_pdfs(rows)                          # downloads from EDGAR if missing (SHA-256 checked)
print("LLM:", llm_name, "| filings:", [p.name for p in pdfs.values()])

## Ingest: text stage only (no OCR, no images)

In [ ]:
WORK.mkdir(parents=True, exist_ok=True)
manifest = WORK / "manifest.jsonl"
manifest.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")
argv = ["ingest", "--root", str(WORK), "--manifest", str(manifest), "--pdf-dir", str(nb.PDF_DIR), "--stages", "text",
        "--ocr", "none", "--embedding-provider", EMBEDDING, "--embedding-model", EMBED_MODEL, "--workers", "1", "--reset"]
argv += ["--max-pages-per-pdf", str(MAX_PAGES)] if MAX_PAGES else []
started = time.time()
ingest = fi.Ingest(fi.build_parser().parse_args(argv))
print(ingest.run(), f"in {time.time() - started:.0f}s")
fi._serving.clear()
fi._serving.update(client=ingest.client, embedder=ingest.embedder)   # reuse the open embedded Qdrant

stats = []
for r in rows:
    pages = json.loads((WORK / "pages" / f"{r['document_id']}.json").read_text(encoding="utf-8"))["pages"]
    stats.append({"filing": r["document_id"], "pages": len(pages), "garbled_pages": sum(p["garbled"] for p in pages),
                  "table_pages": sum(p["page_type"] == "table" for p in pages),
                  "text_points": ingest.client.count(fi.TEXT, count_filter=fi.doc_filter(r["document_id"]), exact=True).count})
nb.show(nb.table(stats))

## Retrieval check: number-aware search
The exact revenue figure from XBRL, typed with or without commas, should find the statement page.

In [ ]:
fact = nb.get_xbrl_fact("AMD", "revenue", 2025)
figure = f"{fact['value'] / 1e6:,.0f}"
for query in [f"total net revenue {figure}", f"net revenue {figure.replace(',', '')}", "AMD net revenue 2025"]:
    hits = fi.retrieve_for_agent(query, top_k=3, ticker="AMD", year=2025)
    print(f"{query!r}:", [(h["page"], h["section"], figure in h["text"]) for h in hits])

## Scenarios × templates × eval questions

In [ ]:
SCENARIOS = {"filtered_k5": (True, 5), "unfiltered_k5": (False, 5), "filtered_k2": (True, 2)}
items = nb.eval_items(DOCS)[: int(os.getenv("NB_QUESTIONS", "0")) or None]


LOADED = {r["document_id"] for r in rows}


def filters_for(item):
    """(ticker, year) of each loaded filing the question is about; comparisons have two."""
    docs = [s["doc_id"] for s in item.get("sources", []) if s.get("doc_id") in LOADED]
    return [(d.split("_")[2], int(d.split("_")[3])) for d in dict.fromkeys(docs)]


def evidence_for(item, filtered, k):
    targets = filters_for(item) if filtered else []
    if not targets:                                  # unfiltered scenario, or a general question
        return fi.retrieve_for_agent(item["question"], top_k=k)
    per = max(1, k // len(targets))
    return [h for ticker, year in targets for h in fi.retrieve_for_agent(item["question"], top_k=per, ticker=ticker, year=year)]


rows_out, replies = [], []
for scenario, (filtered, k) in SCENARIOS.items():
    for item in items:
        hits = nb.label_hits(evidence_for(item, filtered, k))
        for strategy in nb.TEMPLATES:
            reply = nb.run_template(llm, strategy, item["question"], hits)
            result = nb.score(item, reply, hits)
            model, problems = validate_answer(reply["parsed"], hits)
            rows_out.append({"scenario": scenario, "id": item["id"], "category": item["category"], "strategy": strategy,
                             "filing": (filters_for(item) or [(None, None)])[0][0], "correct": result["correct"], "failure": result["failure"],
                             "gold_doc_retrieved": result["gold_doc_retrieved"],
                             "gold_value_in_evidence": result["gold_value_in_evidence"],
                             "grounding_errors": ",".join(problems) if strategy != "zero_shot" else ""})
            replies.append((scenario, item, strategy, hits, reply))
results = nb.table(rows_out)
print(len(results), "answers")

## Results by scenario

In [ ]:
nb.show(results.pivot_table(index="scenario", columns="strategy", values="correct", aggfunc="mean").round(2))
per_item = results.drop_duplicates(["scenario", "id"])
answerable = per_item[per_item.category.isin(["single_figure", "computed_metric", "comparison", "text_fact", "chart_value"])]
nb.show(answerable.groupby("scenario")[["gold_doc_retrieved", "gold_value_in_evidence"]].mean().round(2))
nb.show(results[~results.correct].pivot_table(index="failure", columns="scenario", values="id", aggfunc="count", fill_value=0))

## Garbled filing: Chevron FY2022 against the clean filings (filtered `k=5`)

In [ ]:
base = results[results.scenario == "filtered_k5"].copy()
base["filing_group"] = base.filing.map(lambda t: "CVX (garbled)" if t == "CVX" else ("none" if t is None else "clean"))
nb.show(base.groupby(["filing_group", "strategy"])[["correct", "gold_value_in_evidence"]].mean().round(2))

## Examples of each failure

In [ ]:
seen = set()
for (scenario, item, strategy, hits, reply), row in zip(replies, rows_out):
    key = (row["failure"], scenario)
    if row["failure"] and key not in seen:
        seen.add(key)
        print(f"--- {row['failure']} | {scenario} | {strategy} | {item['id']}: {item['question']}\n"
              f"GOLD: {item['gold_answer'][:110]}\nEVIDENCE: {[(h['doc_id'], h['page']) for h in hits]}\nOUT:  {reply['text'][:300]!r}\n")

## Failure scenarios for text-only RAG

| Failure | Cause | Where it shows up |
|---|---|---|
| **Retrieval miss: wrong company or year** | No filter, so similar wording from another filing ranks higher | `unfiltered_k5`: `gold_doc_retrieved` drops |
| **Retrieval miss: right filing, wrong page** | The figure is in a statement table, but narrative pages rank higher; `k=2` makes this worse | `gold_value_in_evidence` is lower than `gold_doc_retrieved` |
| **Garbled text layer** | Chevron FY2022's statements (pp. 58–62) have a broken font. Their text is never indexed, so the figures can't be retrieved | CVX rows: `gold_value_in_evidence ≈ 0`. Fixed by notebook 03 (OCR) or the 10-K XBRL |
| **Chart-only values** | Numbers printed only inside images | Every `chart_value` item fails here. Fixed by notebooks 03–05 |
| **Wrong row or scale** | Tables flatten into text, so the model picks a neighbouring row or drops "in millions" | `reasoning_error`, `wrong_or_missing_scale` |
| **Unsupported or copied numbers** | The model invents a figure, or copies the few-shot example | `hallucination_unsupported_claim`, `copied_few_shot_example`, `value_not_in_evidence` |
| **Refusal drift** | Advice or a forecast; or over-refusal when the evidence is thin | `permission_routing_failure`, `over_refusal` |